# Price curves 02: noise model

Issue #86 fits the curve families by censored maximum likelihood, which needs a
model for the noise ε in y = h(R) + ε. Two candidates:

- **Constant σ:** ε ~ N(0, σ²). With no censored slots the likelihood is maximised
  by least squares.
- **σ growing with R:** ε ~ N(0, σ(R)²) with σ increasing. With no censored slots
  this is weighted least squares.

This notebook looks at the residuals of **exact slots** against R (their spread
and their tails) to choose between them. Training years only (2021–2024).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from fleet_sim.prices.config import load_config
from fleet_sim.prices.curves import CubicOffset, Isotonic
from fleet_sim.prices.dataset import build_dataset, exact_only, selection_data
from fleet_sim.prices.targets import TARGETS
from fleet_sim.viz.style import apply_style, style_axes

# The config's cache_dir is relative to the repo root, so run from there.
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
apply_style()

## Load

Only exact slots have a residual. Censored slots are counted so it's clear how
much of each year is missing from what follows.

In [ ]:
config = load_config(Path("calibration/price_curves.toml"))
selection = selection_data(build_dataset(config), config.splits)
dataset = exact_only(selection)

obs = dataset.observations
load = np.array([o.residual_load_gw for o in obs])
cost = np.array([o.fuel_cost_gbp_per_mwh for o in obs])
year = np.array([o.year for o in obs])
years = dataset.years
colours = dict(zip(years, plt.get_cmap("viridis")(np.linspace(0, 0.9, len(years)))))

print(f"{'year':>5} {'exact':>6} {'censored':>9}")
for yr in years:
    n_censored = sum(1 for o in selection.observations if o.year == yr and o.censored)
    print(f"{yr:>5} {(year == yr).sum():>6} {n_censored:>9}")

## Residuals

For each target, fit on the exact slots (one fit per year for `raw_per_year`, one
pooled fit otherwise) and take eᵢ = yᵢ − ĥ(Rᵢ).

`Isotonic` is the main curve. It is the most flexible family, so its residuals are
mostly noise rather than a wrong shape. `CubicOffset` is a check that the answer
doesn't depend on the curve.

`raw_pooled` fits one curve to every year, but π moves with k from year to year,
so its residuals mix noise with year-to-year shifts. It is shown for completeness;
`raw_per_year` and `fuel_adjusted` are the ones to read.

In [ ]:
FAMILIES = {"isotonic": Isotonic(), "cubic_offset": CubicOffset()}


def residuals(target_name, family):
    target = TARGETS[target_name]
    y = np.array([target.to_target(o) for o in obs])
    e = np.empty_like(y)
    groups = [[yr] for yr in years] if target.per_year else [years]
    for group in groups:
        m = np.isin(year, group)
        fitted = family.fit(load[m].tolist(), y[m].tolist())
        e[m] = y[m] - np.array([fitted.predict(float(r)) for r in load[m]])
    return e


resid = {
    (t, f): residuals(t, family) for t in TARGETS for f, family in FAMILIES.items()
}

## Spread against R

Bins 1 GW wide, as in notebook 01, keeping bins with at least 30 points. In each
bin: the standard deviation (dashed) and a robust scale, 1.4826 × the median
absolute deviation (solid). The robust scale equals σ for normal data but ignores
a few extreme slots, so the two differ where the tails are heavy.

In [ ]:
BIN_GW = 1.0
MIN_POINTS = 30
edges = np.arange(np.floor(load.min()), np.ceil(load.max()) + BIN_GW, BIN_GW)
centres = (edges[:-1] + edges[1:]) / 2
bin_of = np.digitize(load, edges) - 1


def robust_scale(x):
    return 1.4826 * np.median(np.abs(x - np.median(x)))


def binned_scale(e, mask):
    # Standard deviation and robust scale of e in each R bin; NaN where thin.
    sd = np.full(len(centres), np.nan)
    rs = np.full(len(centres), np.nan)
    for b in range(len(centres)):
        inside = e[mask & (bin_of == b)]
        if len(inside) >= MIN_POINTS:
            sd[b] = inside.std()
            rs[b] = robust_scale(inside)
    return sd, rs


fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, target_name in zip(axes, TARGETS):
    e = resid[(target_name, "isotonic")]
    for yr in years:
        sd, rs = binned_scale(e, year == yr)
        ax.plot(centres, rs, color=colours[yr], label=str(yr))
        ax.plot(centres, sd, color=colours[yr], linestyle="--", linewidth=0.8)
    ax.set_title(target_name)
    ax.set_xlabel("R (GW)")
    style_axes(ax)
axes[0].set_ylabel("residual scale (y units)")
axes[0].legend()
fig.tight_layout()

The same by numbers. For each year: the robust scale below 8 GW, at 10–15 GW and
at 25–30 GW, the ratio of the last two, and the Spearman rank correlation between
|e| and R over all slots. A σ that grows with R would show a ratio well above 1
and a clearly positive correlation in every year.

In [ ]:
def band(rs, lo, hi):
    inside = (centres >= lo) & (centres < hi) & ~np.isnan(rs)
    return rs[inside].mean()


for target_name in TARGETS:
    for family_name in FAMILIES:
        e = resid[(target_name, family_name)]
        print(f"{target_name} / {family_name}")
        print(f"  {'year':>5} {'<8 GW':>7} {'10-15':>7} {'25-30':>7} {'ratio':>6} {'ρ(|e|,R)':>9}")
        for yr in years:
            m = year == yr
            _, rs = binned_scale(e, m)
            low, mid, high = band(rs, -np.inf, 8), band(rs, 10, 15), band(rs, 25, 30)
            rho = stats.spearmanr(load[m], np.abs(e[m])).statistic
            print(
                f"  {yr:>5} {low:>7.3g} {mid:>7.3g} {high:>7.3g} "
                f"{high / mid:>6.2f} {rho:>9.2f}"
            )

### Does the raw spread follow k?

In the raw targets the spread rises with R from 10 to 30 GW in 2022–2024. High R
comes in winter, when k is also high. If ε scales with k, dividing by k (the
fuel-adjusted target) should flatten that rise. Per R bin in 2023 and 2024: the
robust scale of the `raw_per_year` residual and the median k.

In [ ]:
e = resid[("raw_per_year", "isotonic")]
for yr in (2023, 2024):
    m = year == yr
    _, rs = binned_scale(e, m)
    k_median = np.array(
        [np.median(cost[m & (bin_of == b)]) if not np.isnan(rs[b]) else np.nan for b in range(len(centres))]
    )
    ok = ~np.isnan(rs)
    rho_scale = stats.spearmanr(centres[ok], rs[ok]).statistic
    rho_k = stats.spearmanr(centres[ok], k_median[ok]).statistic
    rho_ratio = stats.spearmanr(centres[ok], rs[ok] / k_median[ok]).statistic
    print(f"{yr}: ρ(R, scale) = {rho_scale:.2f}, ρ(R, k) = {rho_k:.2f}, ρ(R, scale / k) = {rho_ratio:.2f}")
    print(f"  scale at 10-15 GW: {band(rs, 10, 15):.1f} £/MWh, k {np.nanmean(k_median[(centres >= 10) & (centres < 15)]):.0f}")
    print(f"  scale at 25-30 GW: {band(rs, 25, 30):.1f} £/MWh, k {np.nanmean(k_median[(centres >= 25) & (centres < 30)]):.0f}")

## Tails

Each residual is standardised two ways: by its R bin's robust scale, and by one
robust scale for the whole year. If heavy tails came only from spread changing
with R, the per-bin version would look normal. For a normal distribution, 4.6%
of points lie beyond 2 and 0.27% beyond 3, and the excess kurtosis is 0.

In [ ]:
def standardised(e, mask, per_bin):
    if not per_bin:
        return e[mask] / robust_scale(e[mask])
    _, rs = binned_scale(e, mask)
    ok = ~np.isnan(rs)
    return e[mask] / np.interp(load[mask], centres[ok], rs[ok])


print(f"{'target':<14} {'year':>5} {'scale':>7} {'>2':>6} {'>3':>7} {'kurt':>6}")
for target_name in TARGETS:
    e = resid[(target_name, "isotonic")]
    for yr in years:
        for per_bin in (True, False):
            z = standardised(e, year == yr, per_bin)
            print(
                f"{target_name:<14} {yr:>5} {'bin' if per_bin else 'year':>7} "
                f"{np.mean(np.abs(z) > 2):>6.3f} {np.mean(np.abs(z) > 3):>7.4f} "
                f"{stats.kurtosis(z):>6.1f}"
            )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True, sharey=True)
for ax, target_name in zip(axes, TARGETS):
    e = resid[(target_name, "isotonic")]
    for yr in years:
        z = np.sort(standardised(e, year == yr, per_bin=True))
        q = stats.norm.ppf((np.arange(z.size) + 0.5) / z.size)
        ax.plot(q, z, color=colours[yr], linewidth=1.0, label=str(yr))
    ax.plot([-4, 4], [-4, 4], color="#1f2933", linewidth=0.8, linestyle=":")
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-8, 8)
    ax.set_title(target_name)
    ax.set_xlabel("normal quantile")
    style_axes(ax)
axes[0].set_ylabel("standardised residual (per bin)")
axes[0].legend()
fig.tight_layout()

## Caveat: censoring cuts off the top

In 2021 and 2022 about a third of slots are censored, and they are the
highest-price slots, mostly at high R. The exact slots left at high R are cut off
from above, so their spread there is understated. This shows as the 2021 raw
spread *falling* with R. 2023 and 2024 have few censored slots (2024 has 5), so
they carry the most weight here. Notebook 01 also found an undetected retail
ceiling from October 2022 to June 2023, which squashes the top of 2022 and 2023.

## Conclusion

**Spread against R.** The spread is not constant, and it doesn't rise steadily
with R. It is U-shaped in every year and target:

- It is largest below about 8 GW: 1.8–2.6× the 10–15 GW level for
  `fuel_adjusted`, and 1.6–1.7× for `raw_per_year` in 2023–2024. That is the
  renewables-heavy regime with negative prices.
- It is lowest from 10 to 20 GW.
- It rises above 20 GW. For `fuel_adjusted` the 25–30 / 10–15 ratio is 1.1–1.2
  in 2023–2024 and 1.6 in 2021–2022. 2022, the gas-crisis year, rises the most
  and nearly doubles by 30 GW. Censoring removes the largest residuals at high R,
  so in 2021–2022 the true rise is if anything bigger than shown.
- Because the low-R and high-R ends pull against each other, the rank
  correlation between |e| and R is weak: −0.15 to 0.16 for `fuel_adjusted`.

**Raw against fuel-adjusted.** For `raw_per_year` the rise from 10–15 to 25–30 GW
is bigger, 1.5–1.9×. Part of it comes from k: in 2023 dividing by k turns a flat
spread into a falling one. In 2024 the spread rises 1.7× while k rises only 1.14×,
so k doesn't explain all of it.

**Tails.** The tails are heavy, mostly for `fuel_adjusted`:

- In 2022–2024, 2.6–3.6% of `fuel_adjusted` slots lie beyond 3 scales (normal:
  0.27%), and the excess kurtosis is 4.7–7.4.
- `raw_per_year` is closer to normal: 0.2–2% beyond 3.
- Standardising per bin barely changes either, so the heavy tails are not a side
  effect of spread changing with R.
- The QQ plots show both tails heavy for `fuel_adjusted`. The lower tail is
  negative prices at low R.

**Choice: constant σ.**

- No single σ that increases with R fits both ends. It would shrink σ at low R,
  where the spread is largest, to gain at high R, where the rise is clear only in
  2021–2022.
- A constant σ fitted jointly lands between the two ends.
- It makes the uncensored fit equal to least squares, which keeps the families'
  existing estimates as the special case with no censoring.

**What it gives up.**

- At high R in 2021–2022, σ is understated, and that is exactly where the
  censored slots are. With σ too small and normal tails too thin, a censored
  slot's term, log P(y ≥ lower bound), is too sure the price sat just above its
  bound. So the fit will tend to place those slots too low. 2021 leans on this
  most.
- A U-shaped or heavier-tailed noise model is out of scope for #86.